# 10-714 Homework 4

In this homework, you will leverage all of the components built in the last three homeworks to solve some modern problems with high performing network structures. We will start by adding a few new ops leveraging our new CPU/CUDA backends. Then, you will implement convolution, and a convolutional neural network to train a classifier on the CIFAR-10 image classification dataset. Then, you will implement recurrent and long-short term memory (LSTM) neural networks, and do word-level prediction language modeling on the Penn Treebank dataset.

As always, we will start by copying this notebook and getting the starting code.
Reminder: __you must save a copy in drive__.

In [1]:
# Mount Drive (same as before)
from google.colab import drive
drive.mount('/content/drive')

# %cd /content/drive/MyDrive/10714

# Clone YOUR repo into a folder named "hw4"
# !git clone https://github.com/3N3G/dlsys-final.git proj
%cd /content/drive/MyDrive/10714/proj

# Same installs as before
!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

Mounted at /content/drive
/content/drive/MyDrive/10714/proj
  Cloning https://github.com/dlsys10714/mugrade.git to /tmp/pip-req-build-ie9nsi8w
  Running command git clone --filter=blob:none --quiet https://github.com/dlsys10714/mugrade.git /tmp/pip-req-build-ie9nsi8w
  Resolved https://github.com/dlsys10714/mugrade.git to commit ac73f725eb2ce0e2c6a38fa540035ee970b8b873
  Preparing metadata (setup.py) ... done
  Created wheel for mugrade: filename=mugrade-1.3-py3-none-any.whl size=3708 sha256=89e02ee51c0d0f413e87b8b6a4162c9f6e8b287c0e8feec30f57198ad96cea74
  Stored in directory: /tmp/pip-ephem-wheel-cache-ng5033mf/wheels/df/c7/14/2b747145fc762900af3ff05bd0c9192c506e70db3ef3890239
Successfully built mugrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 11.1 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

# 1. Load credentials from Secrets
token = userdata.get('GITHUB_TOKEN')
email = userdata.get('GIT_EMAIL')
name = userdata.get('GIT_NAME')
repo_url = "github.com/3N3G/dlsys-final.git" # Your specific repo
repo_name = "dlsys-final"

# 2. Configure Git (Global)
!git config --global user.email "$email"
!git config --global user.name "$name"

In [28]:
!make

CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- Found pybind11: /usr/local/lib/python3.12/dist-packages/pybind11/include (found version "3.0.1")
-- Found cuda, building cuda backend
Fri Dec  5 16:41:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|

In [3]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

env: PYTHONPATH=./python
env: NEEDLE_BACKEND=nd


In [3]:
import sys
sys.path.append('./python')
sys.path.append('./apps')
sys.path.append('./PyTorch_Optimizers')

In [5]:
import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

device = ndl.cuda()
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
dataloader = ndl.data.DataLoader(\
         dataset=dataset,
         batch_size=128,
         shuffle=True,)

model = ResNet9(device=device, dtype="float32")

n_epochs = 30
total_steps = n_epochs * len(dataset)/128
lr1=0.007
lr2=0.2

optimizer_kwargs = dict(
    muon_lr=lr1,
    sgd_lr=lr2,
    momentum=0.95,
    ns_steps=3,
    total_steps=total_steps,
)

train_acc, train_loss = train_cifar10(
    model,
    dataloader,
    n_epochs=30,
    optimizer=ndl.optim.Muon,
    optimizer_kwargs=optimizer_kwargs,
)
evaluate_cifar10(model, dataloader)

Using needle backend
Epoch 00 | train_acc=0.4011, train_loss=1.7300 | test_acc=0.4818, test_loss=1.4316
Epoch 01 | train_acc=0.5254, train_loss=1.3165 | test_acc=0.5653, test_loss=1.2052
Epoch 02 | train_acc=0.5789, train_loss=1.1749 | test_acc=0.5465, test_loss=1.2837
Epoch 03 | train_acc=0.6129, train_loss=1.0798 | test_acc=0.6005, test_loss=1.1183


KeyboardInterrupt: 

In [5]:
# Download the datasets you will be using for this assignment

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

# Download CIFAR-10 dataset
if not os.path.isdir("./data/cifar-10-batches-py"):
    urllib.request.urlretrieve("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", "./data/cifar-10-python.tar.gz")
    !tar -xvzf './data/cifar-10-python.tar.gz' -C './data'

You will now use your convolutional layer to implement a model similar to _ResNet9_, which is known to be a reasonable model for getting good accuracy on CIFAR-10 quickly (see [here](https://github.com/davidcpage/cifar10-fast)). Our main change is that we used striding instead of pooling and divided all of the channels by 4 for the sake of performance (as our framework is not as well-optimized as industry-grade frameworks).

In the figure below, before the first linear layer, you should "flatten" the tensor. You can use the module `Flatten` in `nn_basic.py`, or you can simply use `.reshape` in the `forward()` method of your ResNet9.

Make sure that you pass the device to all modules in your model; otherwise, you will get errors about mismatched devices when trying to run with CUDA.

<center><img src="https://github.com/dlsyscourse/hw4/blob/main/ResNet9.png?raw=true" alt="ResNet9" style="width: 400px;" /></center>

We have tried to make it easier to pass the tests here than for previous assignments where you have implemented models. In particular, we are just going to make sure it has the right number of parameters and similar accuracy and loss after 1 or 2 batches of CIFAR-10.

Now, you can train your model on CIFAR-10 using the following code. Note that this is likely going to be quite slow, and also  not all that accurate due to the lack of data augmentation. You should expect it to take around 500s per epoch.

In [6]:
import sys
sys.path.append('./python')
sys.path.append('./apps')

import importlib
import needle
importlib.reload(needle)
import models
importlib.reload(models)
import simple_ml
importlib.reload(simple_ml)

import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

print("HERE")
device = ndl.cuda()
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
print("LOADING DATA")
dataloader = ndl.data.DataLoader(\
         dataset=dataset,
         batch_size=128,
         shuffle=True,)
print("MODEL")
model = ResNet9(device=device, dtype="float32")
print("TRAINING")
train_cifar10(model, dataloader, n_epochs=30, optimizer=ndl.optim.Adam,
      lr=0.001, weight_decay=0.001)
evaluate_cifar10(model, dataloader)

Using needle backend
HERE
LOADING DATA
MODEL
TRAINING
Epoch 00 | train_acc=0.3897, train_loss=1.6995 | test_acc=0.4813, test_loss=1.4310


KeyboardInterrupt: 

Testing New Optimizers

In [29]:
import sys, importlib
sys.path.append('./python')
sys.path.append('./apps')

import needle
import models
import simple_ml

importlib.reload(needle.optim)
importlib.reload(needle)
importlib.reload(models)
importlib.reload(simple_ml)

import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

try:
    device = ndl.cuda()
except Exception:
    device = ndl.cpu()
print("Using device:", device)

dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
dataloader = ndl.data.DataLoader(
    dataset=dataset,
    batch_size=128,
    shuffle=True,
)

n_epochs = 30
total_steps = n_epochs * len(dataset)/128   # or 8 * len(dataloader) to mimic CifarNet

optim_experiments = [
    ("Muon", ndl.optim.Muon, 0.007, 0.2, 0.0),   # muon_lr, sgd_lr, weight_decay
    # ("SOAP", ndl.optim.SOAP, 0.003, None, 0.0),   # lr, -, wd
]

results = {}

for name, opt_cls, lr1, lr2, wd in optim_experiments:
    print("\n==============================")
    if name == "Muon":
        print(f"  Training with {name}")
        print(f"  muon_lr={lr1}, sgd_lr={lr2}, weight_decay={wd}")
    else:
        print(f"  Training with {name}")
        print(f"  lr={lr1}, weight_decay={wd}")
    print("==============================")

    model = ResNet9(device=device, dtype="float32")

    if name == "Muon":
        optimizer_kwargs = dict(
            muon_lr=lr1,
            sgd_lr=lr2,
            momentum=0.95,
            ns_steps=3,
            weight_decay=wd,
            total_steps=total_steps,   # <-- pass in schedule horizon
        )
        train_acc, train_loss = train_cifar10(
            model,
            dataloader,
            n_epochs=n_epochs,
            optimizer=opt_cls,
            optimizer_kwargs=optimizer_kwargs,
        )
    else:
        train_acc, train_loss = train_cifar10(
            model,
            dataloader,
            n_epochs=n_epochs,
            optimizer=opt_cls,
            lr=lr1,
            weight_decay=wd,
            optimizer_kwargs=None,
        )

    eval_acc, eval_loss = evaluate_cifar10(model, dataloader)

    print(f"[{name}] final train_acc={train_acc:.4f}, train_loss={train_loss:.4f}")
    print(f"[{name}] final eval_acc ={eval_acc:.4f}, eval_loss ={eval_loss:.4f}")

    results[name] = dict(
        train_acc=train_acc,
        train_loss=train_loss,
        eval_acc=eval_acc,
        eval_loss=eval_loss,
    )

print("\nSummary:")
for name, stats in results.items():
    print(
        f"{name:5s} | "
        f"train_acc={stats['train_acc']:.4f}, train_loss={stats['train_loss']:.4f} | "
        f"eval_acc={stats['eval_acc']:.4f}, eval_loss={stats['eval_loss']:.4f}"
    )


Using device: cuda()

  Training with Muon
  muon_lr=0.007, sgd_lr=0.2, weight_decay=0.0
Epoch 00 | train_acc=0.4011, train_loss=1.7300 | test_acc=0.4818, test_loss=1.4316
Epoch 01 | train_acc=0.5254, train_loss=1.3165 | test_acc=0.5653, test_loss=1.2052
Epoch 02 | train_acc=0.5789, train_loss=1.1749 | test_acc=0.5465, test_loss=1.2837
Epoch 03 | train_acc=0.6129, train_loss=1.0798 | test_acc=0.6005, test_loss=1.1183
Epoch 04 | train_acc=0.6441, train_loss=0.9934 | test_acc=0.6389, test_loss=1.0198
Epoch 05 | train_acc=0.6667, train_loss=0.9330 | test_acc=0.6707, test_loss=0.9350
Epoch 06 | train_acc=0.6854, train_loss=0.8854 | test_acc=0.6582, test_loss=0.9589
Epoch 07 | train_acc=0.7014, train_loss=0.8380 | test_acc=0.6706, test_loss=0.9333
Epoch 08 | train_acc=0.7181, train_loss=0.7900 | test_acc=0.6524, test_loss=1.0225
Epoch 09 | train_acc=0.7307, train_loss=0.7533 | test_acc=0.6800, test_loss=0.9305
Epoch 10 | train_acc=0.7466, train_loss=0.7127 | test_acc=0.7047, test_loss=0.848

In [43]:
import sys, importlib
sys.path.append('./python')
sys.path.append('./apps')

import needle
import models
import simple_ml

importlib.reload(needle.optim)
importlib.reload(needle)
importlib.reload(models)
importlib.reload(simple_ml)

import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

try:
    device = ndl.cuda()
except Exception:
    device = ndl.cpu()
print("Using device:", device)

dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
dataloader = ndl.data.DataLoader(
    dataset=dataset,
    batch_size=128,
    shuffle=True,
)

n_epochs = 30
total_steps = n_epochs * len(dataset)/128   # or 8 * len(dataloader) to mimic CifarNet

optim_experiments = [
    # ("Muon", ndl.optim.Muon, 0.007, 0.2, 0.0),   # muon_lr, sgd_lr, weight_decay
    ("SOAP", ndl.optim.SOAP, 0.005, None, None),   # lr, -, wd
    ("SOAP", ndl.optim.SOAP, 0.01, None, None),   # lr, -, wd
]

results = {}

for name, opt_cls, lr1, lr2, wd in optim_experiments:
    print("\n==============================")
    if name == "Muon":
        print(f"  Training with {name}")
        print(f"  muon_lr={lr1}, sgd_lr={lr2}, weight_decay={wd}")
    else:
        print(f"  Training with {name}")
        print(f"  lr={lr1}, weight_decay={wd}")
    print("==============================")

    model = ResNet9(device=device, dtype="float32")

    if name == "Muon":
        optimizer_kwargs = dict(
            muon_lr=lr1,
            sgd_lr=lr2,
            momentum=0.95,
            ns_steps=3,
            weight_decay=wd,
            total_steps=total_steps,   # <-- pass in schedule horizon
        )
        train_acc, train_loss = train_cifar10(
            model,
            dataloader,
            n_epochs=n_epochs,
            optimizer=opt_cls,
            optimizer_kwargs=optimizer_kwargs,
        )
    else:
        optimizer_kwargs = dict(
            lr=lr1,
            total_steps=total_steps,   # <-- pass in schedule horizon
        )
        train_acc, train_loss = train_cifar10(
            model,
            dataloader,
            n_epochs=n_epochs,
            optimizer=opt_cls,
            optimizer_kwargs=optimizer_kwargs,
        )

    eval_acc, eval_loss = evaluate_cifar10(model, dataloader)

    print(f"[{name}] final train_acc={train_acc:.4f}, train_loss={train_loss:.4f}")
    print(f"[{name}] final eval_acc ={eval_acc:.4f}, eval_loss ={eval_loss:.4f}")

    results[name] = dict(
        train_acc=train_acc,
        train_loss=train_loss,
        eval_acc=eval_acc,
        eval_loss=eval_loss,
    )

print("\nSummary:")
for name, stats in results.items():
    print(
        f"{name:5s} | "
        f"train_acc={stats['train_acc']:.4f}, train_loss={stats['train_loss']:.4f} | "
        f"eval_acc={stats['eval_acc']:.4f}, eval_loss={stats['eval_loss']:.4f}"
    )


Using device: cuda()

  Training with SOAP
  lr=0.005, weight_decay=None
Epoch 00 | train_acc=0.4069, train_loss=1.6629 | test_acc=0.4796, test_loss=1.4345


KeyboardInterrupt: 

#Grid Search

In [34]:
import sys, importlib
sys.path.append('./python')
sys.path.append('./apps')

import needle
import models
import simple_ml

importlib.reload(needle.optim)
importlib.reload(needle)
importlib.reload(models)
importlib.reload(simple_ml)

import needle as ndl
import needle.nn as nn
from models import ResNet9
from simple_ml import evaluate_cifar10, epoch_general_cifar10

# Pick device
try:
    device = ndl.cuda()
except Exception:
    device = ndl.cpu()
print("Using device:", device)

# Data
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
dataloader = ndl.data.DataLoader(
    dataset=dataset,
    batch_size=512,
    shuffle=True,
)

# Hyperparameter grids
# For Muon: optimize muon_lr and momentum (and sgd_lr)
muon_muon_lr_grid = [0.007, 0.002, 0.001]
muon_momentum_grid = [0.95]
muon_sgd_lr_grid = [0.2, 0.4]
muon_weight_decay = 0.0

# For SOAP: optimize lr only
soap_lr_grid = [0.007]
soap_weight_decay = 0.0

n_search_epochs = 5
first_epoch_cutoff = 0.25  # if first epoch acc < this, skip rest

results = {
    "Muon": [],
    "SOAP": [],
}

# Small helper to train with early reject
def train_with_early_reject(
    model,
    dataloader,
    optimizer_cls,
    optimizer_kwargs,
    n_epochs,
    cutoff=0.25,
):
    loss_module = nn.SoftmaxLoss()
    opt = optimizer_cls(model.parameters(), **optimizer_kwargs)

    last_train_acc, last_train_loss = 0.0, 0.0

    for epoch in range(n_epochs):
        last_train_acc, last_train_loss = epoch_general_cifar10(
            dataloader, model, loss_fn=loss_module, opt=opt
        )
        print(
            f"  Epoch {epoch:02d} | "
            f"train_acc={last_train_acc:.4f}, train_loss={last_train_loss:.4f}"
        )

        # If first epoch is bad, stop training this config
        if epoch == 0 and last_train_acc < cutoff:
            print(f"  Early reject (first-epoch acc={last_train_acc:.4f} < {cutoff})")
            break

    eval_acc, eval_loss = evaluate_cifar10(model, dataloader)
    return last_train_acc, last_train_loss, eval_acc, eval_loss

# ########################################
# # Grid search: Muon
# ########################################
# print("\n===== Grid search: Muon =====")
# for muon_lr in muon_muon_lr_grid:
#     for sgd_lr in muon_sgd_lr_grid:
#         for mom in muon_momentum_grid:
#             print(f"\n--- Muon: muon_lr={muon_lr}, sgd_lr={sgd_lr}, momentum={mom} ---")

#             model = ResNet9(device=device, dtype="float32")

#             train_acc, train_loss, eval_acc, eval_loss = train_with_early_reject(
#                 model,
#                 dataloader,
#                 optimizer_cls=ndl.optim.Muon,
#                 optimizer_kwargs=dict(
#                     muon_lr=muon_lr,
#                     sgd_lr=sgd_lr,
#                     momentum=mom,
#                     weight_decay=muon_weight_decay,
#                 ),
#                 n_epochs=n_search_epochs,
#                 cutoff=first_epoch_cutoff,
#             )

#             print(
#                 f"[Muon muon_lr={muon_lr}, sgd_lr={sgd_lr}, mom={mom}] "
#                 f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f}, "
#                 f"eval_acc={eval_acc:.4f}, eval_loss={eval_loss:.4f}"
#             )

#             results["Muon"].append({
#                 "muon_lr": muon_lr,
#                 "sgd_lr": sgd_lr,
#                 "momentum": mom,
#                 "train_acc": train_acc,
#                 "train_loss": train_loss,
#                 "eval_acc": eval_acc,
#                 "eval_loss": eval_loss,
#             })

########################################
# Grid search: SOAP
########################################
print("\n===== Grid search: SOAP =====")
for lr in soap_lr_grid:
    print(f"\n--- SOAP: lr={lr} ---")

    model = ResNet9(device=device, dtype="float32")

    train_acc, train_loss, eval_acc, eval_loss = train_with_early_reject(
        model,
        dataloader,
        optimizer_cls=ndl.optim.SOAP,
        optimizer_kwargs=dict(
            lr=lr,
            weight_decay=soap_weight_decay,
        ),
        n_epochs=n_search_epochs,
        cutoff=first_epoch_cutoff,
    )

    print(
        f"[SOAP lr={lr}] "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f}, "
        f"eval_acc={eval_acc:.4f}, eval_loss={eval_loss:.4f}"
    )

    results["SOAP"].append({
        "lr": lr,
        "train_acc": train_acc,
        "train_loss": train_loss,
        "eval_acc": eval_acc,
        "eval_loss": eval_loss,
    })

########################################
# Find best configs by eval_acc
########################################
best_muon = max(results["Muon"], key=lambda r: r["eval_acc"]) if results["Muon"] else None
best_soap = max(results["SOAP"], key=lambda r: r["eval_acc"]) if results["SOAP"] else None

print("\n===== Summary (5-epoch grid search with early reject) =====")

print("\nMuon candidates:")
for r in results["Muon"]:
    print(
        f"muon_lr={r['muon_lr']:.4g}, sgd_lr={r['sgd_lr']:.4g}, mom={r['momentum']:.2f} | "
        f"train_acc={r['train_acc']:.4f}, eval_acc={r['eval_acc']:.4f}"
    )

print("\nSOAP candidates:")
for r in results["SOAP"]:
    print(
        f"lr={r['lr']:.4g} | "
        f"train_acc={r['train_acc']:.4f}, eval_acc={r['eval_acc']:.4f}"
    )

print("\nBest Muon config (by eval_acc):", best_muon)
print("Best SOAP config (by eval_acc):", best_soap)


Using device: cuda()

===== Grid search: SOAP =====

--- SOAP: lr=0.007 ---


KeyboardInterrupt: 

Using Pytorch

Using pytorch resnet 9

In [5]:
# ================================
# 0. Setup & imports
# ================================
!pip install -q torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# 1. CIFAR-10 data loaders
# ================================
transform = T.Compose([
    T.ToTensor(),                     # [0, 1]
    T.Normalize((0.5, 0.5, 0.5),      # mean
                (0.5, 0.5, 0.5)),     # std
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2)

# ================================
# 2. ResNet9 definition (PyTorch)
#    matches your Needle version:
#    - ConvBN(3,16,7,4)
#    - ConvBN(16,32,3,2)
#    - ConvBN(32,32,3,1) x2 + residual
#    - ConvBN(32,64,3,2)
#    - ConvBN(64,128,3,2)
#    - ConvBN(128,128,3,1) x2 + residual
#    - Flatten, Linear(128,128), ReLU, Linear(128,10)
# ================================
class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k, s):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=k, stride=s,
                      padding=k // 2, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResNet9(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Layer 1-8 conv blocks
        self.conv1 = ConvBNReLU(3,   16, 7, 4)
        self.conv2 = ConvBNReLU(16,  32, 3, 2)
        self.conv3 = ConvBNReLU(32,  32, 3, 1)
        self.conv4 = ConvBNReLU(32,  32, 3, 1)

        self.conv5 = ConvBNReLU(32,  64, 3, 2)
        self.conv6 = ConvBNReLU(64, 128, 3, 2)
        self.conv7 = ConvBNReLU(128,128, 3, 1)
        self.conv8 = ConvBNReLU(128,128, 3, 1)

        # Fully connected part
        # After conv6 on CIFAR-10 with these strides, spatial size is 1x1,
        # so feature dim = 128.
        self.fc1 = nn.Linear(128, 128)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Input: (N, 3, 32, 32)
        x = self.conv1(x)
        out2 = self.conv2(x)

        x = self.conv3(out2)
        x = self.conv4(x)

        # Residual from layer 2
        x = x + out2

        x = self.conv5(x)
        out6 = self.conv6(x)

        x = self.conv7(out6)
        x = self.conv8(x)

        # Flatten both x and out6, add residual
        x_flat   = x.view(x.size(0), -1)      # (N, 128 * 1 * 1) = (N, 128)
        out6_flat = out6.view(out6.size(0), -1)
        x = x_flat + out6_flat

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# ================================
# 3. Loss & optimizer
# ================================
criterion = nn.CrossEntropyLoss()

# ================================
# 4. Train / eval helpers
# ================================
def train_one_epoch(model, loader, optimizer, criterion, device, scheduler=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        # Optional per-step LR scheduler
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total



@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


Using device: cuda


In [5]:
# ================================
# 5. Main training loop
# ================================
model = ResNet9().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    train_acc, train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_acc, test_loss   = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f} | "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


Epoch 01 | train_acc=0.4358, train_loss=1.5541 | test_acc=0.4874, test_loss=1.4155
Epoch 02 | train_acc=0.5316, train_loss=1.2956 | test_acc=0.5452, test_loss=1.2681


KeyboardInterrupt: 

In [13]:
import importlib
import Muon
importlib.reload(Muon)
from Muon import Muon
from torch.optim.lr_scheduler import LambdaLR

model = ResNet9().to(device)
criterion = nn.CrossEntropyLoss()

total_steps = num_epochs * len(train_loader)
optimizer = Muon(
    model.parameters(),
    muon_lr=0.007,
    sgd_lr=0.2,
    momentum=0.6,
    nesterov=True,
    total_steps=total_steps,
    weight_decay=0.01,
)

for epoch in range(1, num_epochs + 1):
    train_acc, train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
    )
    test_acc, test_loss = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f} | "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


[Muon] Epoch 01 | train_acc=0.4036, train_loss=1.6398 | test_acc=0.4924, test_loss=1.4046
[Muon] Epoch 02 | train_acc=0.5295, train_loss=1.3031 | test_acc=0.5340, test_loss=1.2945
[Muon] Epoch 03 | train_acc=0.5886, train_loss=1.1522 | test_acc=0.5576, test_loss=1.2500
[Muon] Epoch 04 | train_acc=0.6304, train_loss=1.0402 | test_acc=0.5653, test_loss=1.2294
[Muon] Epoch 05 | train_acc=0.6686, train_loss=0.9424 | test_acc=0.5701, test_loss=1.2325
[Muon] Epoch 06 | train_acc=0.7002, train_loss=0.8601 | test_acc=0.5732, test_loss=1.2439
[Muon] Epoch 07 | train_acc=0.7304, train_loss=0.7836 | test_acc=0.5744, test_loss=1.2599
[Muon] Epoch 08 | train_acc=0.7545, train_loss=0.7167 | test_acc=0.5741, test_loss=1.2939
[Muon] Epoch 09 | train_acc=0.7773, train_loss=0.6580 | test_acc=0.5739, test_loss=1.3127
[Muon] Epoch 10 | train_acc=0.7970, train_loss=0.6108 | test_acc=0.5731, test_loss=1.3234


In [ ]:
from SOAP import SOAP
import torch.nn as nn

model = ResNet9().to(device)
criterion = nn.CrossEntropyLoss()

optimizer = SOAP(
    model.parameters(),
    lr=0.005,
)

n_epochs = 10

for epoch in range(1, n_epochs + 1):
    train_acc, train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
    )
    test_acc, test_loss = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f} | "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


Epoch 01 | train_acc=0.4731, train_loss=1.4510 | test_acc=0.5475, test_loss=1.2527
Epoch 02 | train_acc=0.5893, train_loss=1.1427 | test_acc=0.5996, test_loss=1.1182
Epoch 03 | train_acc=0.6509, train_loss=0.9847 | test_acc=0.6343, test_loss=1.0306
